# Konvolúcia a filtrácia

Doplnte príklady vlastnými poznámkami o ich fungovaní.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
import pywt
import skimage as sk
from scipy.signal import convolve2d, correlate2d

image = sk.img_as_float(sk.data.camera())     # 512 x 512, hodnoty v [0, 1]

## 1. Vlastná 2D konvolúcia

Z definície $y(n_1,n_2)=\sum_{k_1}\sum_{k_2} h(k_1,k_2)\,x(n_1-k_1,\,n_2-k_2)$ vyplýva, že výstup je **vážený súčet posunutých kópií obrazu**: pre každý prvok jadra $h(k_1,k_2)$ posunieme obraz o $(k_1,k_2)$, vynásobíme ho týmto prvkom a pripočítame. Slučka teda ide cez prvky jadra (9 pre 3×3), nie cez pixely obrazu.

Režim *same* a nulový padding ako `convolve2d(..., mode='same', boundary='fill')`.

In [ ]:
def konvolucia_2d(x, h):
    P, Q = h.shape
    cr, cs = (P - 1) // 2, (Q - 1) // 2               # stred jadra (rovnako ako scipy pri 'same')
    xp = np.pad(x, ((P - 1 - cr, cr), (Q - 1 - cs, cs)))   # nulový padding
    M, N = x.shape
    y = np.zeros((M, N))
    for k1 in range(P):
        for k2 in range(Q):
            posunuty = xp[P - 1 - k1 : P - 1 - k1 + M,    # x(n1 - k1, n2 - k2)
                          Q - 1 - k2 : Q - 1 - k2 + N]
            y += h[k1, k2] * posunuty
    return y

h = np.array([[1, 2, 1],
              [2, 4, 2],
              [1, 2, 1]]) / 16                         # Gauss 3x3 z prednášky
print("zhoda so scipy:", np.allclose(konvolucia_2d(image, h),
                                     convolve2d(image, h, mode="same", boundary="fill")))

### Konvolúcia vs. korelácia

Konvolúcia jadro preklápa ($x(n-k)$), korelácia nie ($x(n+k)$). Pri symetrickom jadre je to jedno, pri nesymetrickom nie.

In [ ]:
sobel_x = np.array([[-1, 0, 1],
                    [-2, 0, 2],
                    [-1, 0, 1]])

konv = convolve2d(image, sobel_x, mode="same", boundary="symm")
kor = correlate2d(image, sobel_x, mode="same", boundary="symm")
print("konvolúcia == korelácia:   ", np.allclose(konv, kor))
print("konvolúcia == -korelácia:  ", np.allclose(konv, -kor))
print("Gauss: konvolúcia == korelácia:",
      np.allclose(convolve2d(image, h, mode="same"), correlate2d(image, h, mode="same")))

## 2. Väzba na DWT: Haarov krok je konvolúcia + decimácia

Jeden krok DWT po riadkoch = konvolúcia s 1D filtrom a ponechanie každej druhej vzorky. Pri konvolúcii (s preklopením) treba použiť filtre tak, ako ich uvádza pywt: `dec_hi` = $[-\tfrac{\sqrt2}{2}, \tfrac{\sqrt2}{2}]$, teda preklopené $g$ z prednášky.

In [ ]:
def haar_krok_riadky(img):
    pass # z casti cvicenia A

w = pywt.Wavelet("haar")
h_lo = np.array(w.dec_lo)[None, :]                   # 1 x 2 jadro
h_hi = np.array(w.dec_hi)[None, :]

c_konv = convolve2d(image, h_lo, mode="full")[:, 1::2]   # filtrácia + decimácia
d_konv = convolve2d(image, h_hi, mode="full")[:, 1::2]

print("zhoda s haar_krok_riadky:",
      np.allclose(np.hstack([c_konv, d_konv]), haar_krok_riadky(image)))

## 3. Separovateľnosť

Jadro je separovateľné práve vtedy, keď má ako matica hodnosť 1 (dá sa zapísať ako vonkajší súčin dvoch vektorov).

Doplňte meranie času podľa príkladu:

t = time.perf_counter()

funkcia ktorej čas výpočtu meriame

cas_funkcie = time.perf_counter() - t   # cas teraz - cas na zaciatku

Takto porovnajte dĺžku separovaného a neseparovaného výpočtu.

In [ ]:
from scipy.signal.windows import gaussian
import time

g1 = gaussian(21, std=5.0)
g1 /= g1.sum()
h_gauss = np.outer(g1, g1)                           # 21 x 21

y_2d = convolve2d(image, h_gauss, mode="same", boundary="symm")

y_sep = convolve2d(image, g1[None, :], mode="same", boundary="symm")   # po riadkoch
y_sep = convolve2d(y_sep, g1[:, None], mode="same", boundary="symm")   # po stĺpcoch

print("zhoda:", np.allclose(y_2d, y_sep))
print(f"cas vypoctu: ")

for nazov, jadro in [("Gauss 21x21", h_gauss),
                     ("Sobel x", sobel_x),
                     ("Laplace", np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]]))]:
    print(f"{nazov:12s} hodnosť = {np.linalg.matrix_rank(jadro)}")

## 4. Frekvenčná charakteristika jadier

Podľa $Y = H\,X$ je účinok filtra daný amplitúdou $|H|$. Jadro doplníme nulami na 64×64 a spravíme 2D DFT.

In [ ]:
def frekv_char(jadro, N=64):
    return np.abs(np.fft.fftshift(np.fft.fft2(jadro, s=(N, N))))

jadra = {
    "priemer 3x3": np.ones((3, 3)) / 9,
    "Gauss 3x3":   np.array([[1, 2, 1], [2, 4, 2], [1, 2, 1]]) / 16,
    "Laplace":     np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]]),
    "Sobel x":     sobel_x,
}

fig, ax = plt.subplots(1, len(jadra), figsize=(16, 4))
for a, (nazov, jadro) in zip(ax, jadra.items()):
    im = a.imshow(frekv_char(jadro), cmap="viridis", extent=[-0.5, 0.5, 0.5, -0.5])
    a.set_title(nazov)
    a.set_xlabel("frekvencia [cykly/pixel]")
    fig.colorbar(im, ax=a, shrink=0.8)
plt.tight_layout()
plt.show()

## 5. Interaktívne skúšanie 3×3 jadier

Zmena čísla v mriežke hneď prepočíta výsledok aj frekvenčnú charakteristiku. Hornopriepustné jadrá dávajú záporné hodnoty, preto je na výber spôsob zobrazenia.

In [ ]:
PRESETY = {
    "identita":  [[0, 0, 0], [0, 1, 0], [0, 0, 0]],
    "posun":     [[1, 0, 0], [0, 0, 0], [0, 0, 0]],
    "priemer":   [[1/9] * 3] * 3,
    "Gauss":     [[1/16, 2/16, 1/16], [2/16, 4/16, 2/16], [1/16, 2/16, 1/16]],
    "ostrenie":  [[0, -1, 0], [-1, 5, -1], [0, -1, 0]],
    "Laplace":   [[0, 1, 0], [1, -4, 1], [0, 1, 0]],
    "Sobel x":   [[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]],
    "Sobel y":   [[-1, -2, -1], [0, 0, 0], [1, 2, 1]],
}

def zobrazitelne(y, rezim):
    """Prevedie výstup filtra na hodnoty vhodné pre imshow."""
    if rezim == "orezať do [0, 1]":
        return np.clip(y, 0, 1), dict(cmap="gray", vmin=0, vmax=1)
    if rezim == "absolútna hodnota":
        return np.abs(y), dict(cmap="gray")
    m = np.percentile(np.abs(y), 99) + 1e-12          # symetricky okolo 0, orezané na 99. percentil
    return y, dict(cmap="RdBu_r", vmin=-m, vmax=m)

bunky = [[widgets.FloatText(layout=widgets.Layout(width="70px")) for _ in range(3)]
         for _ in range(3)]
preset = widgets.Dropdown(options=list(PRESETY), value="identita", description="preset")
rezim = widgets.Dropdown(options=["orezať do [0, 1]", "absolútna hodnota", "symetricky okolo 0"],
                         description="zobrazenie")
okraj = widgets.Dropdown(options=["symm", "fill", "wrap"], description="okraj")
vystup = widgets.Output()
nacitavam = False                                      # počas načítania presetu neprepočítavať 9-krát

def nastav_preset(change=None):
    global nacitavam
    nacitavam = True
    for i in range(3):
        for j in range(3):
            bunky[i][j].value = PRESETY[preset.value][i][j]
    nacitavam = False
    prepocitaj()

def prepocitaj(change=None):
    if nacitavam:
        return
    jadro = np.array([[bunky[i][j].value for j in range(3)] for i in range(3)])
    y = convolve2d(image, jadro, mode="same", boundary=okraj.value)
    data, kw = zobrazitelne(y, rezim.value)
    with vystup:
        vystup.clear_output(wait=True)
        fig, ax = plt.subplots(1, 3, figsize=(15, 5), gridspec_kw=dict(width_ratios=[2, 2, 1.4]))
        ax[0].imshow(image, cmap="gray", vmin=0, vmax=1); ax[0].set_title("originál")
        ax[1].imshow(data, **kw)
        ax[1].set_title(f"výsledok, súčet jadra = {jadro.sum():.3g}, "
                        f"rozsah [{y.min():.2f}, {y.max():.2f}]")
        ax[2].imshow(frekv_char(jadro), cmap="viridis", extent=[-0.5, 0.5, 0.5, -0.5])
        ax[2].set_title("|H|")
        for a in ax[:2]:
            a.axis("off")
        plt.show()

preset.observe(nastav_preset, names="value")
for w_ in [rezim, okraj] + [b for riadok in bunky for b in riadok]:
    w_.observe(prepocitaj, names="value")

nastav_preset()
mriezka = widgets.VBox([widgets.HBox(riadok) for riadok in bunky])
display(widgets.HBox([mriezka, widgets.VBox([preset, rezim, okraj])]), vystup)

## 6. Lineárny vs. nelineárny filter pri impulznom šume

Doplňte premenné zasumeny (random_noise), gauss (convolve2d) a median (median_filter).
Implementujte aj minimálny a maximálny filter podľa prednášky.

In [ ]:
from scipy.ndimage import median_filter

zasumeny = ...
gauss = ...
median = ...

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
for a, obr, nazov in zip(ax, [zasumeny, gauss, median],
                         ["soľ a korenie 5 %", "Gauss 3x3", "medián 3x3"]):
    a.imshow(obr[150:350, 150:350], cmap="gray", vmin=0, vmax=1)
    a.set_title(nazov)
    a.axis("off")
plt.tight_layout()
plt.show()